<a href="https://colab.research.google.com/github/shahedtheboss/SECOM_kaggle_comp/blob/main/SECOM_kaggle_comp.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
%pip install -q pyarrow

In [11]:
import pandas as pd
import numpy as np
import kagglehub
from pathlib import Path
import pyarrow.parquet as pq

In [ ]:
kagglehub.login()


Kaggle credentials set.
Kaggle credentials successfully validated.


In [12]:
path = kagglehub.competition_download(
    'secom-defect-detection-challenge'
)

print(path)

/root/.cache/kagglehub/competitions/secom-defect-detection-challenge


In [ ]:
competition_dir = Path(path)


for item in competition_dir.iterdir():
  print(item)

/root/.cache/kagglehub/competitions/secom-defect-detection-challenge/test.csv
/root/.cache/kagglehub/competitions/secom-defect-detection-challenge/sample_submission.csv
/root/.cache/kagglehub/competitions/secom-defect-detection-challenge/train.csv


In [ ]:
train = pd.read_csv(competition_dir / 'train.csv')
print("Training data loaded!")
test = pd.read_csv(competition_dir / 'test.csv')
print("Test data loaded!")

sample_submission = pd.read_csv(competition_dir / 'sample_submission.csv')
print("Sample Data loaded!")

Training data loaded!
Test data loaded!
Sample Data loaded!


In [ ]:
print("Train Shape:", train.shape)
print("Test Shape:", test.shape)
print("Sample Shape:", sample_submission.shape)
# print("Training Schema:")
# print(train.schema)

# print("Test Shape:", test.shape)
# print("Sample Submission Shape:", sample_submisssion.shape)

Train Shape: (1253, 593)
Test Shape: (314, 592)
Sample Shape: (314, 2)


In [ ]:
print(train.schema.names)

['ingest_lib', 'normalized_smiles', 'inchikey', 'inchikey14', 'molecular_formula', 'ionization_mode', 'instrument_type', 'adduct', 'adduct_orig', 'precursor_mz', 'precursor_error_ppm', 'element', 'element', 'num_peaks', 'base_peak_intensity', 'element', 'collision_energy_orig', 'collision_energy_orig_units']


In [ ]:

print("TRAIN COLUMNS:")
print(train.columns.tolist())

print("\nTEST COLUMNS:")
print(test.columns.tolist())

print("\nSAMPLE SUBMISSION COLUMNS:")
print(sample_submission.columns.tolist())

TRAIN COLUMNS:
['id', 'time', '0', '1', '2', '3', '4', '5', '6', '7', '8', '9', '10', '11', '12', '13', '14', '15', '16', '17', '18', '19', '20', '21', '22', '23', '24', '25', '26', '27', '28', '29', '30', '31', '32', '33', '34', '35', '36', '37', '38', '39', '40', '41', '42', '43', '44', '45', '46', '47', '48', '49', '50', '51', '52', '53', '54', '55', '56', '57', '58', '59', '60', '61', '62', '63', '64', '65', '66', '67', '68', '69', '70', '71', '72', '73', '74', '75', '76', '77', '78', '79', '80', '81', '82', '83', '84', '85', '86', '87', '88', '89', '90', '91', '92', '93', '94', '95', '96', '97', '98', '99', '100', '101', '102', '103', '104', '105', '106', '107', '108', '109', '110', '111', '112', '113', '114', '115', '116', '117', '118', '119', '120', '121', '122', '123', '124', '125', '126', '127', '128', '129', '130', '131', '132', '133', '134', '135', '136', '137', '138', '139', '140', '141', '142', '143', '144', '145', '146', '147', '148', '149', '150', '151', '152', '153', '1

In [ ]:
display(train.head())


In [ ]:
# Data types and non-null counts
print("TRAIN INFO:")
train.info()

print("\nTEST INFO:")
test.info()

TRAIN INFO:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1253 entries, 0 to 1252
Columns: 593 entries, id to target
dtypes: float64(590), int64(2), object(1)
memory usage: 5.7+ MB

TEST INFO:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 314 entries, 0 to 313
Columns: 592 entries, id to 589
dtypes: float64(590), int64(1), object(1)
memory usage: 1.4+ MB


In [ ]:
print("Train missing values:")
print(train.isnull().sum())

print("\nTest missing values:")
print(test.isnull().sum())

Train missing values:
id         0
time       0
0          5
1          5
2         13
          ..
586        1
587        1
588        1
589        1
target     0
Length: 593, dtype: int64

Test missing values:
id      0
time    0
0       1
1       2
2       1
       ..
585     0
586     0
587     0
588     0
589     0
Length: 592, dtype: int64


In [ ]:
selected_columns = ['ingest_lib', 'normalized_smiles', 'inchikey', 'inchikey14', 'molecular_formula', 'ionization_mode', 'instrument_type', 'adduct', 'adduct_orig', 'precursor_mz', 'precursor_error_ppm', 'element', 'element', 'num_peaks', 'base_peak_intensity', 'element', 'collision_energy_orig', 'collision_energy_orig_units']


batches = train.iter_batches(
    batch_size=5,
    columns=selected_columns
)

first_batch = next(batches)
small_sample = first_batch.to_pandas()

display(small_sample)


,ingest_lib,normalized_smiles,inchikey,inchikey14,molecular_formula,ionization_mode,instrument_type,adduct,adduct_orig,precursor_mz,precursor_error_ppm,num_peaks,base_peak_intensity,collision_energy_orig,collision_energy_orig_units
0,drug_plus,O=C1NC(=O)c2cc(Nc3ccccc3)c(Nc3ccccc3)cc21,AAALVYBICLMAMA-UHFFFAOYSA-N,AAALVYBICLMAMA,C20H15N3O2,positive,None,[M+H]+,[M+H]+,330.1237,1.671398,52,NaN,None,unknown
1,drug_plus,C#Cc1cccc(Nc2ncnc3cc(OCCOC)c(OCCOC)cc23)c1,AAKJLRGGTJKAMG-UHFFFAOYSA-N,AAKJLRGGTJKAMG,C22H23N3O4,positive,None,[M+K]+,[M+K]+,432.1320,1.302193,28,NaN,None,unknown
2,drug_plus,CC1(C)CCC(C)(C)c2cc(C(O)C(O)=Nc3ccc(C(=O)O)cc3...,AANFHDFOMFRLLR-UHFFFAOYSA-N,AANFHDFOMFRLLR,C23H26FNO4,positive,None,[M+Na]+,[M+Na]+,422.1738,1.316437,41,NaN,None,unknown
3,drug_plus,CN1C(=O)CN=C(c2ccccc2)c2cc(Cl)ccc21,AAOVKJBEBIDNHE-UHFFFAOYSA-N,AAOVKJBEBIDNHE,C16H13ClN2O,positive,None,[M+H]+,[M+H]+,285.0789,1.984579,166,NaN,None,unknown
4,drug_plus,NC(=O)C1=CN(C2OC(COP(=O)(O)OP(=O)(O)OCC3OC(n4c...,ACFIXJIJDZMPPO-UHFFFAOYSA-N,ACFIXJIJDZMPPO,C21H30N7O17P3,positive,None,[M+H]+,[M+H]+,746.0984,0.708429,186,NaN,None,unknown


In [ ]:
rows_inspected = 0

for batch in train.iter_batches(
    batch_size=1000,
    columns=selected_columns
):
 rows_inspected += batch.num_rows

 if rows_inspected >= 5000:
  break


print(rows_inspected)

5000


In [ ]:
sample_submission = pd.read_csv(
    competition_dir / 'sample_submission.csv',
    nrows=5
    )
display(sample_submission)
print(sample_submission.columns.tolist())

,molecule_id,smiles
0,m_005e53,CCO;CCO;CCO;CCO;CCO;CCO;CCO;CCO;CCO;CCO;CCO;CC...
1,m_006153,CCO;CCO;CCO;CCO;CCO;CCO;CCO;CCO;CCO;CCO;CCO;CC...
2,m_00b5aa,CCO;CCO;CCO;CCO;CCO;CCO;CCO;CCO;CCO;CCO;CCO;CC...
3,m_01e922,CCO;CCO;CCO;CCO;CCO;CCO;CCO;CCO;CCO;CCO;CCO;CC...
4,m_0259d4,CCO;CCO;CCO;CCO;CCO;CCO;CCO;CCO;CCO;CCO;CCO;CC...


['molecule_id', 'smiles']
